# Meta-Learned Personalization of Insulin Therapy in Type 1 Diabetes

## Load and Inspect Data

In [3]:
import os
import pandas as pd

# Relative path from the notebook to the CSV files
data_dir = "../data"
patient_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

# Dictionary to store data for each patient
patient_data = {}

# Load and parse each CSV
for file in sorted(patient_files):
    file_path = os.path.join(data_dir, file)
    patient_id = file.split('.')[0]

    # Load CSV and parse timestamp
    df = pd.read_csv(file_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.sort_values(by='timestamp').reset_index(drop=True)

    # Store in dictionary
    patient_data[patient_id] = df

print(f"Loaded {len(patient_data)} patient files.")

C:\Users\snand\AppData\Local\Temp\ipykernel_14380\1614044804.py:17: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
C:\Users\snand\AppData\Local\Temp\ipykernel_14380\1614044804.py:17: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Loaded 20 patient files.


In [4]:
# Basic inspection of each patient's DataFrame
summary = []

for patient_id, df in patient_data.items():
    total_rows = len(df)
    total_missing = df.isnull().sum().sum()
    missing_percent = (total_missing / (total_rows * df.shape[1])) * 100

    summary.append({
        'Patient ID': patient_id,
        'Total Rows': total_rows,
        'Start Time': df['timestamp'].min(),
        'End Time': df['timestamp'].max(),
        'Total Missing Values': total_missing,
        'Missing %': round(missing_percent, 2)
    })

# Create and display summary DataFrame
summary_df = pd.DataFrame(summary)
print(summary_df)

# Save to CSV
summary_df.to_csv("../output/combined_inspection_summary.csv", index=False)


   Patient ID  Total Rows          Start Time            End Time  \
0         P01      136220 2023-06-01 00:05:00 2023-12-20 15:30:00   
1         P02      174856 2023-06-01 00:01:00 2023-12-27 09:20:00   
2         P03       82602 2023-06-01 00:03:00 2023-09-06 11:25:00   
3         P04      183068 2023-06-06 00:02:00 2024-01-10 21:25:00   
4         P05      148201 2023-06-01 00:05:00 2024-01-09 18:00:00   
5         P06      132444 2023-06-01 00:05:00 2023-12-18 10:35:00   
6         P07      128824 2023-06-01 00:05:00 2023-12-27 10:33:00   
7         P10      165048 2023-06-01 00:03:00 2023-12-22 14:35:00   
8         P11      167251 2023-06-01 00:01:19 2024-01-08 16:45:00   
9         P12      176293 2023-06-01 00:03:00 2024-01-10 13:05:00   
10        P13       63310 2023-06-01 00:05:00 2023-09-03 22:00:00   
11        P15      167535 2023-06-01 00:05:00 2024-01-12 17:46:00   
12        P16      162371 2023-06-05 02:11:00 2023-12-20 10:55:00   
13        P17      110866 2023-06-

### Overview
A total of 20 patient data files (P01 - P24, excluding some IDs) from the BrisT1D dataset were loaded and inspected. Each file contains time-series data with features including timestamped blood glucose level (`bg`), insulin dosage, carbohydrate intake, and additional physiological metrics.

#### Time Span
* All patients have data starting from **early June 2023**, with most records continuing into **December 2023 or January 2024**.
* The dataset spans over **6-7 months of continuous glucose monitoring** per patient, which is sufficient for both training robust time-series models and simulating real-world personalization scenarios.

#### Data Density
* The **number of rows per patient** varies from ~63000 (P13) to ~178000 (P24), indicating slight differences in recording frequency or duration.
* This variability supports the use of **meta-learning**, where each patient acts as an individual task, possibly with different data volumes and distributions.

#### Missing Data
* The percentage of missing data ranges from **~59.7% (P01)** to **~69.9% (P18)**.
* This high proportion of missingness is expected in wearable CGM and insulin tracking data due to:
    * Device disconnections
    * Logging inconsistencies
* **Imputation strategies** will be necessary during preprocessing.

#### Implications for Modeling
* The combination of:
    * **High Missingness**
    * **Inter-patient variability**
makes this dataset ideal for testing both:
* Traditional time-series models
* **Meta-learning approaches**


## Preprocessing and Feature Standardization
### Preprocessing 
* Convert timestamps to datetime format and set them as the index.
* Filter out non-numeric columns.
* Drop features with more than **90% missing values**.
* Resample each patient to **5-Minute Intervals** to ensure temporal consistency.
* Apply **time-based interpolation** and **forward/backward filling** to handle gaps.
* Save cleaned data to `../output` directory.

### Feature Standardization
* Identifies all unique numeric features across patients.
* Ensures every patient has the same set of features (`bg`, `cals`, `dist`, `hr`, `insulin`, `steps`).
* Missing features are added as NaNs and filled using forward/backward fill.

In [6]:
def preprocess_patient_df(df):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp').sort_index()

    # Select numeric columns only
    df_numeric = df.select_dtypes(include='number')

    # Drop features with >90% missing
    df_numeric = df_numeric.dropna(thresh=int(0.1 * len(df_numeric)), axis=1)

    # Resample to 5-minute intervals
    df_resampled = df_numeric.resample("5min").mean()

    # Interpolate and fill
    df_interpolated = df_resampled.interpolate(method="time", limit_direction="both")
    df_filled = df_interpolated.ffill().bfill()

    return df_filled.reset_index()
    
preprocessed_data = {}

for patient_id, df in patient_data.items():
    preprocessed_data[patient_id] = preprocess_patient_df(df)

# check post-processing
for pid, df in preprocessed_data.items():
    missing = df.isnull().sum().sum()
    if missing > 0:
        print(f"{pid}: Still has {missing} missing values.")

In [7]:
# Descriptive Statistics
preprocessed_data['P01'].describe()

,timestamp,bg,insulin,hr,dist,steps,cals
count,58362,58362.000000,58362.000000,58362.000000,58362.000000,58362.000000,58362.000000
mean,2023-09-10 07:47:29.999999744,8.807550,0.135304,77.717772,33.381239,48.781365,8.672061
min,2023-06-01 00:05:00,2.200000,0.000000,45.800000,0.000000,0.000000,0.960000
25%,2023-07-21 15:56:15,5.700000,0.029200,64.700000,0.000000,0.000000,4.800000
50%,2023-09-10 07:47:30,8.100000,0.033300,76.700000,3.900000,5.781746,5.680000
75%,2023-10-30 23:38:45,11.266667,0.058300,87.900000,31.500000,46.500000,10.370000
max,2023-12-20 15:30:00,27.800000,11.741700,172.400000,716.300000,766.000000,53.000000
std,NaN,3.915529,0.713540,15.488071,72.611302,104.460820,6.121277


In [8]:
print(preprocessed_data['P01'].columns.tolist())

['timestamp', 'bg', 'insulin', 'hr', 'dist', 'steps', 'cals']


In [9]:
# Output directory
output_dir = "../output"

# Create directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save each patient's data
for patient_id, df in preprocessed_data.items():
    output_path = os.path.join(output_dir, f"{patient_id}.csv")
    df.to_csv(output_path, index=False)

print(f"Saved cleaned data for {len(preprocessed_data)} patients to {output_dir}")

Saved cleaned data for 20 patients to ../output


In [10]:
import numpy as np

# Get all unique numeric features across patients
all_features = set()
for df in preprocessed_data.values():
    all_features.update(col for col in df.columns if col != 'timestamp')

all_features = sorted(all_features)
print("All available features:", all_features)

# Define required features (manually or use all)
required_features = all_features  # or ['bg', 'insulin', 'hr', 'dist', 'steps', 'cals']

# Standardize feature set and fill missing values
standardized_data = {}

for patient_id, df in preprocessed_data.items():
    std_df = df[['timestamp']].copy()  # Start with timestamp column

    # Ensure all required features exist and are in the same order
    for feature in required_features:
        if feature in df.columns:
            std_df[feature] = df[feature]
        else:
            std_df[feature] = np.nan  # Add missing feature as NaN

    # Fill missing values (using updated syntax to avoid warnings)
    std_df = std_df.ffill().bfill()

    standardized_data[patient_id] = std_df

print("Standardization complete. All patients now have the same features.")

# Confirm consistency
for pid, df in standardized_data.items():
    features = [col for col in df.columns if col != 'timestamp']
    print(f"{pid}: {features}")

All available features: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
Standardization complete. All patients now have the same features.
P01: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P02: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P03: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P04: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P05: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P06: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P07: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P10: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P11: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P12: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P13: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P15: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P16: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P17: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P18: ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
P19: ['bg', 'cals', 'dist', 'hr', 'insulin', 'step

### Window Generation
* **Input window sizes**: 6h, 6h, 12h
* **Prediction Offset**: 1h ahead (i.e., predict the value 1 hour after the end of the input window)

#### Assumptions
* Have data at **5-minute intervals** $\rightarrow$ 12 steps/hour.
* So
    * 6h $\rightarrow$ 72 steps
    * 9h $\rightarrow$ 108 steps
    * 12h $\rightarrow$ 144 steps
* Wanting to **predict glucose** at `t+1h`

In [12]:
def generate_windows(
    df,
    input_steps,
    pred_offset,
    target_col='bg',
    allow_missing_target=True
):

    X, y = [], []
    data = df.copy().reset_index(drop=True)

    if target_col not in data.columns:
        raise ValueError(f"Target column '{target_col}' not found in DataFrame")

    for i in range(len(data) - input_steps - pred_offset):
        x_window = data.iloc[i:i + input_steps].drop(columns=['timestamp']).values
        y_target = data.iloc[i + input_steps + pred_offset - 1][target_col]
        
        if pd.isna(y_target) and not allow_missing_target:
            continue
        X.append(x_window)
        y.append(y_target)

    return np.array(X), np.array(y)

In [13]:
import numpy as np

# Apply to all patients and window sizes
input_durations = {
    '6h': 72,
    '9h': 108,
    '12h': 144
}
pred_offset = 12  # predict 1 hour ahead

windowed_data = {}

for duration, steps in input_durations.items():
    windowed_data[duration] = {}
    for patient_id, df in standardized_data.items():
        try:
            
            X, y = generate_windows(
                df,
                input_steps=steps,
                pred_offset=pred_offset,
                target_col='bg',
                allow_missing_target=True
            )
            windowed_data[duration][patient_id] = {'X': X, 'y': y}
            print(f"{duration} - {patient_id}: X = {X.shape}, y = {y.shape}, NaN targets = {np.isnan(y).sum()}")
        except Exception as e:
            print(f"Error processing {duration} - {patient_id}: {e}")


6h - P01: X = (58278, 72, 6), y = (58278,), NaN targets = 0
6h - P02: X = (60221, 72, 6), y = (60221,), NaN targets = 0
6h - P03: X = (27990, 72, 6), y = (27990,), NaN targets = 0
6h - P04: X = (62958, 72, 6), y = (62958,), NaN targets = 0
6h - P05: X = (64068, 72, 6), y = (64068,), NaN targets = 0
6h - P06: X = (57643, 72, 6), y = (57643,), NaN targets = 0
6h - P07: X = (60234, 72, 6), y = (60234,), NaN targets = 0
6h - P10: X = (58844, 72, 6), y = (58844,), NaN targets = 0
6h - P11: X = (63766, 72, 6), y = (63766,), NaN targets = 0
6h - P12: X = (64298, 72, 6), y = (64298,), NaN targets = 0
6h - P13: X = (27252, 72, 6), y = (27252,), NaN targets = 0
6h - P15: X = (64929, 72, 6), y = (64929,), NaN targets = 0
6h - P16: X = (57046, 72, 6), y = (57046,), NaN targets = 0
6h - P17: X = (58377, 72, 6), y = (58377,), NaN targets = 0
6h - P18: X = (64317, 72, 6), y = (64317,), NaN targets = 0
6h - P19: X = (58582, 72, 6), y = (58582,), NaN targets = 0
6h - P21: X = (60858, 72, 6), y = (60858

In [14]:
# Saving to disk
import os

output_dir = "../windowed_data"
os.makedirs(output_dir, exist_ok=True)

for duration, patients in windowed_data.items():
    duration_dir = os.path.join(output_dir, duration)
    os.makedirs(duration_dir, exist_ok=True)

    for patient_id, data in patients.items():
        X = data['X']
        y = data['y']
        filename = f"{patient_id}.npz"
        filepath = os.path.join(duration_dir, filename)

        np.savez_compressed(filepath, X=X, y=y)

print(" All windowed data saved successfully.")

 All windowed data saved successfully.
